# Sparse Router Calibration (Colab)
Calibrate routing and lifecycle behavior with replay + failure simulation.


In [ ]:
import subprocess, pathlib, os
repo_root = pathlib.Path('/content/That-Ai-Coder')
if not repo_root.exists():
    subprocess.run(['git','clone','https://github.com/YOUR_ORG/That-Ai-Coder.git', str(repo_root)], check=False)
os.chdir(repo_root)


In [ ]:
from sparse_autosec.system import SparseExpertAutoSec
from sparse_autosec.config import AutoSecConfig

cfg = AutoSecConfig()
system = SparseExpertAutoSec(cfg)

tasks = [
    'analyze eval injection in parser',
    'analyze pickle loads endpoint',
    'analyze subprocess shell true command',
]
for i in range(30):
    task = tasks[i % len(tasks)]
    complexity = system.core.score_task_complexity(task)
    decision = system.router.route(system.core.encode_task(task), complexity=complexity)
    signature = 'py_eval_user_input' if 'eval' in task else ('unsafe_deserialization' if 'pickle' in task else 'shell_injection')
    system.memory.counters[signature] = system.memory.counters.get(signature, 0) + 1
    system.learning.record_replay(task, decision.selected[0], 1.0 if i % 2 == 0 else 0.0)

lifecycle = system.lifecycle.step()
print('spawned:', lifecycle.spawned)
print('quiesced:', lifecycle.quiesced)
print('retired:', lifecycle.retired)


In [ ]:
for name in system.experts.names(active_only=False):
    ex = system.experts.get(name)
    print(name, ex.state.value, ex.health(), ex.use_count, ex.success_count, ex.fail_count)
